## PoI Plotter — VCO Model Exploration & Design Tool

This notebook provides an **interactive visualization and design tool** for the VCO-based conductance sensing system.

Its main purpose is to:

* **Visualize the full signal chain** from $(G, i_{dc}, f_s)$ to $(\Delta G, P)$
* **Expose all intermediate variables and equations**
* **Understand sensitivities and trade-offs**
* **Solve the inverse problem**: find optimal operating points from target specifications

 The tool combines **forward modeling + reverse optimization** in a single interactive dashboard 

---

## Core Functionality

The plotter allows you to:

### 1. Forward analysis (physics understanding)

Given:

* Conductance $G$
* Bias current $i_{dc}$
* Sampling frequency $f_s$

It computes and visualizes:

* $V_{in}$, $f_{osc}$, $K_{VCO}$
* Frequency error contributions ($\Delta f_{sampling}$, $\Delta f_{adev}$)
* Voltage error $\Delta V_{in}$
* Conductance resolution $\Delta G$
* Power consumption ($P_{idc}, P_{VCO}, P_{CNT}, P_{TOT}$)

 This corresponds to the **physical signal chain** implemented in `forward_compute()` 

---

### 2. Reverse design (control & optimization)

Given:

* Target resolution $\Delta G_{\text{target}}$
* Maximum power $P_{max}$

The tool:

* Sweeps all valid $i_{dc}$
* Computes $(\Delta G, P_{tot})$ for each point
* Builds the feasible region:
  $$
  \Delta G \le \Delta G_{\text{target}}, \quad P_{tot} \le P_{max}
  $$
* Selects the optimal current:
  $$
  i_{dc,opt} = \max(\text{feasible } i_{dc})
  $$

Implemented in `reverse_compute()` 

---

## Interactive Dashboard

The UI is split into two control panels:

### Forward Controls

* **$G$ (μS)**: imposed conductance
* **$i_{dc}$ (μA)**: operating current (bounded automatically)
* **$f_s$ (Hz)**: sampling frequency

### Reverse Controls

* **$\Delta G_{\text{target}}$ (nS)**: desired resolution
* **$P_{max}$ (μW)**: power constraint
* **Variance toggle**: include/exclude Allan deviation

The sliders dynamically adapt (e.g. valid $\Delta G$ range) 

---

## Visualization Panels

The dashboard displays a **6-panel system view**:

### 1. VCO Transfer Curve

* Measured data + model
* Operating point highlighted

### 2. Inputs & Intermediate Values

* $V_{in}$, $f_{osc}$, $K_{VCO}$, $\Delta V_{in}$, $\Delta f_{osc}$

### 3. Frequency Error Breakdown

* Sampling vs Allan deviation contributions

### 4. Output Metrics

* $\Delta G$ (nS)
* Power contributions (μW)

### 5. Trade-off Curve ($i_{dc}$ sweep)

* $\Delta G$ vs $i_{dc}$
* Power vs $i_{dc}$
* Feasible region (green)
* Optimal point (highlighted) 

### 6. Summary Panel

* Current operating point
* Achievable $\Delta G$ range
* Optimal solution (if feasible)

---

## Model Details

### VCO Representation

Two interchangeable models:

* **Polynomial model** (smooth, analytical)
* **LUT model** (data-driven, more accurate)

Controlled via `representation` in `VCO_Model` 

---

### Frequency Error Model

Total frequency uncertainty:
$$
\Delta f_{osc} = \max(\Delta f_{sampling}, \Delta f_{adev})
$$

* **Sampling error**:
  $$
  \Delta f_{sampling} = f_s \cdot \sqrt{avg\_window}
  $$

* **Allan deviation** (optional):

  * extracted from measured oscillator data
  * depends on $V_{in}$ and $\tau = 1/f_s$

---

### Conductance Resolution

Computed from the full nonlinear model:
$$
\Delta G = \left|\frac{\Delta f_{osc} \cdot G^2}{K_{VCO} \cdot i_{dc} + \Delta f_{osc} \cdot G}\right|
$$

---

## How to Use the Tool

### Understanding the system

* Fix $G$, vary $i_{dc}$ and $f_s$
* Observe how $\Delta G$ and power evolve
* Identify optimal regions manually

### Designing the system

* Set:

  * $\Delta G_{\text{target}}$
  * $P_{max}$
* Let the tool:

  * compute feasible region
  * highlight optimal $i_{dc}$

### Key insight

* Increasing $i_{dc}$:

  * improves sensitivity (↓ $\Delta G$)
  * reduces total power (in this regime)
* The optimal point is typically:
    **largest feasible $i_{dc}$**



In [1]:
import sys
import os
from PoI_plotter import *

# Resolve path to vendor VCO model
vendor_vco_path = os.path.abspath(
    os.path.join(os.getcwd(),
    '../../hw/vendor/analog-library/VCO/VCO_characteristics')
)
sys.path.insert(0, vendor_vco_path)
from vco_model import VCOADCModel

import warnings
warnings.filterwarnings('ignore')

## VCO_Model class

In [2]:
# instantiate the model with our data
VCO_model = VCOADCModel()

In [3]:
VCO_model.vin_active

array([330., 340., 360., 380., 400., 420., 440., 460., 480., 500., 520.,
       540., 560., 580., 600., 620., 640., 660., 680., 700., 720., 740.,
       760., 780., 800., 820.])

In [ ]:
PoI_plotter(VCO_model, variance=3)

## Control Algorithm

### Phase 1: Calibration

#### 1. Coarse baseline measurement of $G$

Goal: quickly obtain an initial estimate $\hat G$ over a wide range.

* **Setting:** high G range + high SNR + high $P_{tot}$ + high $\Delta G$ 
* **Point of interest:** high $f_s$, low $i_{dc}$

This gives a first rough estimate:
$$
\hat G_{\text{base}}
$$

#### 2. Refined operating-point selection

Goal: use $\hat G_{\text{base}}$ to choose a better current for the next measurement.

Steps:

1. Use $\hat G_{\text{base}}$ as the conductance estimate.
2. Compute $\Delta G$ ranges with measured G
3. Find $i_{dc}$ corresponding to min $\Delta G$

---

### Phase 2: Measure $G$ at the selected operating point

Once $i_{dc,\mathrm{opt}}$ is chosen:

1. Measure the oscillator count and estimate frequency:
   $$
   f_{osc} = \Delta N \cdot f_s
   $$

2. Recover the corresponding input voltage $V_{in}$:
   This is done by LUT inversion:

   * search the LUT for the closest $f_{osc}$
   * interpolate to obtain $V_{in}$

3. Compute the updated conductance esimate $\hat G$:
   $$
   \hat G = \frac{i_{dc}}{V_{DD} - V_{in}}
   $$


4. Raise an interrupt and **jump to Phase 3** if one of these conditions is violated:
$$
\Delta G = \left|\frac{\Delta f_{osc} \cdot G^2}{K_{VCO} \cdot i_{dc} + \Delta f_{osc} \cdot G}\right| > \Delta G_{\text{target}}
$$
$$
P_{tot} = (i_{dc} \cdot V_{dd} - \frac{i_{dc}^2}{G} )+ P_{VCO} + P_{CNT} > P_{\max}
$$

5. Repeat Phase 2 at sampling frequency rate $f_s$

---

### Phase 3: Compute optimal $i_{dc}$

#### Inputs

* baseline estimate $\hat G$
* target resolution $\Delta G_{\text{target}}$
* sampling frequency $f_s$
* optional power constraint:
  $$
  P_{\mathrm{tot}} \le P_{\max}
  $$

#### Step 1: Compute valid current range

The valid operating current is bounded by:
$$
0 < i_{dc} \le i_{dc,\max}
$$
with
$$
i_{dc,\max} = G \cdot (V_{DD} - V_{\min})
$$

#### Step 2: Sweep the valid $i_{dc}$ range

For each candidate $i_{dc}$, the corresponding $V_{in}$, $K_{VCO}$, $\Delta f$, $\Delta G$, and $P_{tot}$ are computed.

1. Compute the operating voltage:
   $$
   V_{in} = V_{DD} - \frac{i_{dc}}{G}
   $$

2. Find the VCO slope $K_{VCO}$ corresponding to $V_{in}$ using the derivative LUT.

3. Compute the frequency uncertainty:

   * quantization error only:
     $$
     \Delta f_{\mathrm{osc,q}} = f_s
     $$
     
   * if Allan deviation is enabled then:
     $$
     \Delta f_{osc} = \max(\Delta f_{\mathrm{osc,q}}, \Delta f_{\mathrm{adev}})
     $$

4. Compute the conductance resolution $\Delta G$:
   $$
   \Delta G
   = \left|\frac{\Delta f_{osc} \cdot G^2}{K_{VCO} \cdot i_{dc} + \Delta f_{osc} \cdot G}\right|
   $$

5. Compute total power:
   $$
   P_{\mathrm{tot}} = P_{idc} + P_{VCO} + P_{CNT}
   $$
   with
   $$
   P_{idc} = V_{in} \cdot i_{dc} = i_{dc} \cdot V_{dd} - \frac{i_{dc}^2}{G}
   $$
6. A candidate is immediately rejected if one of the following conditions is violated:
   $$
   V_{in} \notin [V_{\min}, V_{DD}]
   $$
   $$
   \Delta G > \Delta G_{\text{target}}
   $$
   $$
   P_{tot} > P_{\max}
   $$


#### Step 3: Choose optimal current

Since the current candidates are scanned in increasing order and, in the considered operating regime, the total power decreases with $i_{dc}$, the optimal current is obtained by keeping the last valid candidate found during the scan. However the higher $i_{dc}$, the lower the range of G, so we need to leave a margin for G to change.

Therefore, the control algorithm does not need to explicitly construct the feasible set. It is sufficient to update the stored solution each time a valid candidate is encountered. At the end of the scan:

* if at least one valid candidate was found, the stored value is used as $i_{dc,\mathrm{opt}}$,
* otherwise the requested $\Delta G_{\text{target}}$ and $P_{\max}$ cannot be satisfied simultaneously.

